In [1]:
! pip install -q schedule pytest # установка библиотек, если ещё не

In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
import json

from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, asdict
from typing import List

from bs4 import BeautifulSoup, PageElement

## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
@dataclass
class BookData:
    name: str
    description: str | None
    upc: str
    product_type: str
    price_excl_tax: str
    price_incl_tax: str
    tax: str
    rating: int
    availablility: str
    availability_number: int
    reviews: int


def get_book_data(book_url: str) -> BookData:
    """
    Получение данных о книге с одной страницы.
    book_url - ссылка на web страницу с книгой.
    """
    response = requests.get(book_url)
    if response.status_code != 200:
        raise RuntimeError(
            f"Failed get request to {book_url} with status {response.status_code}"
        )

    try:
        book_data = parse_book_data(response.text)
    except Exception as ex:
        print(f"Failed to parse book with url {book_url}")
        raise ex

    return book_data


def parse_book_data(html: str) -> BookData:
    soup = BeautifulSoup(html, "html.parser")

    product_info_table = soup.find("table", class_="table table-striped")
    product_info = parse_product_info_table(product_info_table)

    upc = product_info["UPC"]
    tax = product_info["Tax"]
    availability = product_info["Availability"]
    availability_number = int(availability.split("(")[1].split()[0])
    reviews = int(product_info["Number of reviews"])
    product_type = product_info["Product Type"]
    price_excl_tax = product_info["Price (excl. tax)"]
    price_incl_tax = product_info["Price (incl. tax)"]

    product_page_article = soup.find("article", class_="product_page")
    description_p = product_page_article.find("p", recursive=False)
    description_text = description_p.text if description_p else None

    product_main_div = soup.find("div", class_="product_main")
    name = product_main_div.find("h1").text
    rating_p = product_main_div.find("p", class_="star-rating")
    rating = parse_rating(rating_p)

    return BookData(
        name=name,
        description=description_text,
        upc=upc,
        product_type=product_type,
        tax=tax,
        price_excl_tax=price_excl_tax,
        price_incl_tax=price_incl_tax,
        rating=rating,
        availablility=availability,
        availability_number=availability_number,
        reviews=reviews,
    )


def parse_rating(element: PageElement) -> int | None:
    """
    Парсим рейтинг из product_main div.
    """

    class_to_rating_map = {
        "Zero": 0,
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5,
    }

    classes = element.get("class", [])
    for class_name, value in class_to_rating_map.items():
        if class_name in classes:
            return value

    return None


def parse_product_info_table(table: PageElement) -> dict:
    return {item.find("th").text: item.find("td").text for item in table.find_all("tr")}


In [4]:
# Используйте для самопроверки
book_url = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
get_book_data(book_url)

BookData(name='A Light in the Attic', description="It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I think someone do

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
THREAD_POOL_WORKERS = 32
DUMP_FILE_NAME = "books_data.txt"


def scrape_books(is_save: bool) -> List[BookData]:
    """
    Парсим все книжки с books.toscrape.com.
    """
    pages_number = get_pages_number()
    book_urls = get_all_book_urls(pages_number)

    with ThreadPoolExecutor(max_workers=THREAD_POOL_WORKERS) as executor:
        books_data = list(executor.map(get_book_data, book_urls))

    if is_save:
        with open(DUMP_FILE_NAME, "+w") as file:
            json.dump(
                [asdict(b) for b in books_data], file, ensure_ascii=False, indent=4
            )

    return books_data


def get_pages_number():
    """
    Получение количества страниц в каталоге.
    """
    url = get_page_url(1)
    response = requests.get(url)
    if response.status_code != 200:
        raise RuntimeError(f"Failed get request to {url}")

    soup = BeautifulSoup(response.text, "html.parser")
    parts = soup.find("li", class_="current").text.split()
    return int(parts[-1])


def get_all_book_urls(pages_number):
    """
    Собираем список всех ссылок на страницы с книжками.
    pages_number - общее количество страниц.
    """
    all_book_urls = []
    with ThreadPoolExecutor(max_workers=THREAD_POOL_WORKERS) as executor:
        page_numbers = [i for i in range(1, pages_number + 1)]
        book_urls = list(executor.map(get_book_urls_from_page, page_numbers))

    for urls in book_urls:
        all_book_urls.extend(urls)

    return all_book_urls


def get_book_urls_from_page(page_number) -> List[str]:
    """
    Получение списка урлов на все книжки со страници с номером page_number.
    """
    page_url = get_page_url(page_number)
    response = requests.get(page_url)
    if response.status_code != 200:
        raise RuntimeError(
            f"Failed get request to {page_url} with number {page_number}"
        )

    products_href = parse_products_href_in_html(html=response.text)
    return [get_book_url(href) for href in products_href]


def parse_products_href_in_html(html: str) -> List[str]:
    """
    Парсит ссылки на продукты из html отдельной страницы с продуктами из каталога.
    """
    soup = BeautifulSoup(html, "html.parser")
    li_tags = soup.find("ol", class_="row").find_all(
        "li", class_="col-xs-6 col-sm-4 col-md-3 col-lg-3"
    )
    hrefs = []
    for li in li_tags:
        a_tag = li.find("h3").find("a")
        href = a_tag.get("href")
        hrefs.append(href)
    return hrefs


def get_page_url(page_number: int) -> str:
    return f"http://books.toscrape.com/catalogue/page-{page_number}.html"


def get_book_url(book_href: str) -> str:
    return f"https://books.toscrape.com/catalogue/{book_href}"

In [6]:
# Проверка работоспособности функции
books = scrape_books(is_save=True)
print(type(books), len(books))

<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
START_SCHEDULER_LOCAL_TIME = "19:00"
CHECK_PENDING_TIMEOUT_S = 60


def collect_books_data():
    print("Start books data collecting")

    start_time = time.time()
    books_data = scrape_books(is_save=True)
    end_time = time.time()

    print(
        f"Successfully collected {len(books_data)} books by {end_time - start_time:.2f} seconds"
    )


schedule.every().day.at(START_SCHEDULER_LOCAL_TIME).do(collect_books_data)
print(f"Setup scheduler at {START_SCHEDULER_LOCAL_TIME}")

while True:
    schedule.run_pending()
    time.sleep(CHECK_PENDING_TIMEOUT_S)

Setup scheduler at 18:49
Start books data collecting
Successfully collected 1000 books by 22.85 seconds


KeyboardInterrupt: 

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [15]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest ../tests/

============================= test session starts ==============================
platform darwin -- Python 3.12.6, pytest-8.4.2, pluggy-1.6.0
rootdir: /Users/mishasdk/mipt/mipt-py-scrapper
collected 3 items                                                              

../tests/test_scraper.py ...                                             [100%]

============================== 3 passed in 0.09s ===============================


## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```